# Bài học 16 - Triển khai Đại lý Có thể Mở rộng với Microsoft Foundry

Trong sổ tay này, bạn sẽ xây dựng một **đại lý hỗ trợ khách hàng sẵn sàng cho sản xuất** cho công ty hư cấu **Contoso**. Khác với các bài học trước, điểm chính ở đây không phải là vòng lặp suy luận của đại lý — mà là tất cả những gì bao quanh *vòng lặp* đó giúp một đại lý an toàn để chạy ở quy mô lớn:

1. **Gọi công cụ** — tra cứu đơn hàng và tạo vé hỗ trợ.
2. **RAG** — trả lời chính sách từ cơ sở kiến thức.
3. **Bộ nhớ** — ghi nhớ khách hàng qua các lượt tương tác.
4. **Chuyển tuyến mô hình** — gửi yêu cầu đơn giản tới mô hình nhỏ, yêu cầu phức tạp tới mô hình lớn.
5. **Bộ nhớ đệm phản hồi** — phục vụ các câu hỏi lặp lại mà không gọi mô hình.
6. **Phê duyệt của con người** — các khoản hoàn tiền vượt ngưỡng sẽ tạm dừng chờ phê duyệt.
7. **Cổng đánh giá** — bộ kiểm tra ngoại tuyến chặn bản phát hành không tốt.
8. **Khả năng quan sát** — theo dõi OpenTelemetry quanh mỗi yêu cầu.

Mỗi phần độc lập và có thể chạy được. Đọc từng dòng — các nguyên thủy sản xuất được giữ rất nhỏ gọn.


## Thiết lập

Trước khi chạy notebook này, hãy đảm bảo bạn đã:

1. **Một dự án Microsoft Foundry** với mô hình trò chuyện đã được triển khai (ví dụ: `gpt-4.1-mini`).
2. **Đăng nhập bằng Azure CLI** — chạy `az login` trong terminal của bạn.
3. **Đặt các biến môi trường cần thiết:**
   - `AZURE_AI_PROJECT_ENDPOINT` — điểm cuối dự án Microsoft Foundry của bạn.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — tên của mô hình bạn đã triển khai.

Phần RAG sử dụng **Azure AI Search** khi `AZURE_SEARCH_SERVICE_ENDPOINT` và `AZURE_SEARCH_API_KEY` được thiết lập, và sẽ chuyển sang tìm kiếm trong bộ nhớ trong để notebook có thể chạy mà không cần tài nguyên Search.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import re
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME in your .env file."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential(),
)
print("Foundry client ready.")

## 1. Công cụ

Công cụ sản xuất thực hiện công việc thực tế trên các hệ thống thực. Ở đây chúng ta mô phỏng một cơ sở dữ liệu đơn hàng và một hệ thống bán vé bằng các hàm Python thuần túy. Bộ trang trí `@tool` giúp chúng được phơi bày ra tác nhân.

Lưu ý `issue_refund` sử dụng `approval_mode="always_require"` cho các khoản hoàn tiền vượt quá ngưỡng — đây là đại nguyên thủy có con người trong luồng mà chúng ta triển khai sau này.


In [ ]:
# Simulated backend systems (in production these are API calls behind scoped identities).
ORDERS = {
    "A1001": {"status": "shipped", "total": 42.00, "eta": "2 days"},
    "A1002": {"status": "processing", "total": 128.50, "eta": "5 days"},
    "A1003": {"status": "delivered", "total": 19.99, "eta": "delivered"},
}
TICKETS: list[dict] = []
REFUND_APPROVAL_THRESHOLD = 50.0


@tool(approval_mode="never_require")
def get_order_status(order_id: Annotated[str, "The customer's order ID, e.g. A1001"]) -> str:
    """Look up the status of a customer order."""
    order = ORDERS.get(order_id.upper())
    if not order:
        return f"No order found with ID {order_id}."
    return (
        f"Order {order_id.upper()}: status={order['status']}, "
        f"total=${order['total']:.2f}, eta={order['eta']}."
    )


@tool(approval_mode="never_require")
def open_ticket(
    subject: Annotated[str, "Short subject line for the support ticket"],
    details: Annotated[str, "Full description of the customer's issue"],
) -> str:
    """Open a support ticket for issues that need human follow-up."""
    ticket_id = f"T{1000 + len(TICKETS) + 1}"
    TICKETS.append({"id": ticket_id, "subject": subject, "details": details})
    return f"Ticket {ticket_id} opened: {subject}"


def refund_needs_approval(amount: float) -> bool:
    """Refunds above the threshold require a human approver."""
    return amount > REFUND_APPROVAL_THRESHOLD


@tool(approval_mode="always_require")
def issue_refund(
    order_id: Annotated[str, "The order to refund"],
    amount: Annotated[float, "Refund amount in USD"],
) -> str:
    """Issue a refund. Execution pauses for human approval before it runs."""
    return f"Refund of ${amount:.2f} issued for order {order_id.upper()}."


print("Tools defined.")

## 2. RAG — Cơ sở tri thức về Chính sách

Các câu hỏi về chính sách ("thời gian hoàn trả của bạn là bao lâu?") nên được trả lời từ một nguồn đáng tin cậy, không phải từ trí nhớ của mô hình. Chúng tôi bao bọc một cơ sở tri thức nhỏ như một công cụ tìm kiếm.

Trong môi trường sản xuất, đây là **Azure AI Search**; ở đây chúng tôi cung cấp một công cụ tìm kiếm từ khóa trong bộ nhớ để notebook có thể chạy ở mọi nơi, và tự động chuyển sang Azure AI Search khi có các biến môi trường.


In [ ]:
KNOWLEDGE_BASE = {
    "returns": "Contoso accepts returns within 30 days of delivery for a full refund. Items must be unused and in original packaging.",
    "shipping": "Standard shipping takes 3-5 business days. Express shipping (1-2 days) is available at checkout for an extra fee.",
    "warranty": "All Contoso electronics carry a 12-month limited warranty covering manufacturing defects.",
    "refund_policy": "Refunds are processed to the original payment method within 5 business days of approval. Refunds over $50 require a supervisor's approval.",
}


def _in_memory_search(query: str) -> str:
    q = query.lower()
    hits = [text for key, text in KNOWLEDGE_BASE.items() if key.replace("_", " ") in q or key in q]
    if not hits:
        # crude keyword fallback so the tool still returns something useful
        hits = [text for text in KNOWLEDGE_BASE.values() if any(w in text.lower() for w in q.split())]
    return "\n".join(hits) if hits else "No matching policy found."


def _azure_search(query: str) -> str:
    from azure.core.credentials import AzureKeyCredential
    from azure.search.documents import SearchClient

    client = SearchClient(
        endpoint=os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"],
        index_name=os.getenv("AZURE_SEARCH_INDEX_NAME", "contoso-policies"),
        credential=AzureKeyCredential(os.environ["AZURE_SEARCH_API_KEY"]),
    )
    results = client.search(search_text=query, top=3)
    return "\n".join(r.get("content", "") for r in results) or "No matching policy found."


USE_AZURE_SEARCH = bool(os.getenv("AZURE_SEARCH_SERVICE_ENDPOINT") and os.getenv("AZURE_SEARCH_API_KEY"))


@tool(approval_mode="never_require")
def search_policies(query: Annotated[str, "The policy question to look up"]) -> str:
    """Search Contoso support policies to answer customer questions."""
    if USE_AZURE_SEARCH:
        return _azure_search(query)
    return _in_memory_search(query)


print(f"RAG ready. Using {'Azure AI Search' if USE_AZURE_SEARCH else 'in-memory search'}.")

## 3. Bộ nhớ

Một nhân viên hỗ trợ mà quên mất mình đang nói chuyện với ai là một nhân viên hỗ trợ kém. Chúng tôi giữ một bộ hồ sơ nhỏ cho mỗi khách hàng và chèn một bản tóm tắt ngắn vào hướng dẫn của nhân viên. Trong môi trường sản xuất, đây là một dịch vụ bộ nhớ (xem Bài học 13); trong đây một dict làm cho mẫu dễ nhìn thấy.


In [ ]:
CUSTOMER_MEMORY: dict[str, dict] = {
    "cust-42": {"name": "Dana", "tier": "enterprise", "recent_order": "A1002"},
    "cust-99": {"name": "Sam", "tier": "standard", "recent_order": "A1003"},
}


def memory_context(customer_id: str) -> str:
    profile = CUSTOMER_MEMORY.get(customer_id)
    if not profile:
        return "This is a new customer with no history."
    return (
        f"Customer {profile['name']} ({profile['tier']} tier). "
        f"Most recent order: {profile['recent_order']}."
    )


print(memory_context("cust-42"))

## 4 & 5. Định tuyến Mô hình và Bộ nhớ đệm Phản hồi

Hai cần điều chỉnh chi phí được kết nối vào một trình xử lý yêu cầu duy nhất:

- **Định tuyến**: một bộ phân loại heuristic rẻ tiền quyết định xem một yêu cầu cần mô hình nhỏ hay mô hình lớn.
- **Bộ nhớ đệm**: các câu hỏi lặp lại đã được chuẩn hóa được phục vụ trực tiếp từ bộ nhớ đệm mà không gọi mô hình.

Bộ phân loại ở đây được thiết kế đơn giản. Trong môi trường sản xuất, bạn sẽ kiểm tra nó với lưu lượng truy cập và có thể thay thế nó bằng Bộ định tuyến Mô hình của Foundry.


In [ ]:
SMALL_MODEL = os.getenv("AZURE_AI_SMALL_MODEL", model)   # e.g. gpt-4.1-mini
LARGE_MODEL = os.getenv("AZURE_AI_LARGE_MODEL", model)   # e.g. gpt-4.1

response_cache: dict[str, str] = {}
route_counters = {"small": 0, "large": 0, "cache": 0}


def normalize(query: str) -> str:
    return re.sub(r"\s+", " ", query.lower().strip())


COMPLEX_SIGNALS = ("refund", "cancel", "complaint", "escalate", "broken", "wrong", "why")


def is_simple(query: str) -> bool:
    """Route complex or high-stakes requests to the large model; everything else to the small one."""
    q = query.lower()
    if any(signal in q for signal in COMPLEX_SIGNALS):
        return False
    return len(q.split()) <= 20


def choose_model(query: str) -> str:
    return SMALL_MODEL if is_simple(query) else LARGE_MODEL


print(f"Small model: {SMALL_MODEL} | Large model: {LARGE_MODEL}")

## 6 & 8. Tác nhân, Phê duyệt của Con người và Khả năng quan sát

Bây giờ chúng ta lắp ráp tác nhân từ các công cụ ở trên và bao bọc mỗi yêu cầu trong một đoạn OpenTelemetry. Hàm `handle_support_request` là trình xử lý yêu cầu sản xuất: cache → route → trace → run → cache.

Phê duyệt của con người được xử lý bởi khung làm việc: vì `issue_refund` có `approval_mode="always_require"`, quá trình chạy sẽ tạm dừng và hiển thị yêu cầu phê duyệt mà chúng ta sẽ giải quyết một cách rõ ràng.


In [ ]:
# Tracing: use the Agent Framework tracer if available, else a no-op so the notebook runs anywhere.
try:
    from agent_framework.observability import get_tracer
    tracer = get_tracer()
except Exception:  # observability extras not installed
    from contextlib import contextmanager

    class _NoopSpan:
        def set_attribute(self, *_args, **_kwargs):
            pass

    class _NoopTracer:
        @contextmanager
        def start_as_current_span(self, _name):
            yield _NoopSpan()

    tracer = _NoopTracer()


SUPPORT_INSTRUCTIONS = (
    "You are Contoso's customer support agent. Be concise, friendly, and accurate. "
    "Use search_policies for policy questions, get_order_status for orders, "
    "open_ticket when a human needs to follow up, and issue_refund for refunds. "
    "Never invent policy details."
)

support_agent = provider.as_agent(
    name="ContosoSupportAgent",
    instructions=SUPPORT_INSTRUCTIONS,
    tools=[get_order_status, open_ticket, search_policies, issue_refund],
)


async def handle_support_request(query: str, customer_id: str) -> str:
    # 1. Serve from cache when we can.
    key = normalize(query)
    if key in response_cache:
        route_counters["cache"] += 1
        return response_cache[key]

    # 2. Route by complexity to control cost.
    chosen_model = choose_model(query)
    route_counters["small" if chosen_model == SMALL_MODEL else "large"] += 1

    # 3. Add per-customer memory to the prompt.
    context = memory_context(customer_id)
    prompt = f"[Customer context: {context}]\n\n{query}"

    # 4. Run inside a trace span for observability.
    with tracer.start_as_current_span("support_request") as span:
        span.set_attribute("customer.id", customer_id)
        span.set_attribute("routed.model", chosen_model)
        response = await support_agent.run(prompt, model=chosen_model)

    text = response.text
    response_cache[key] = text
    return text


print("Support agent assembled.")

In [ ]:
# Try a few requests. The first is simple (small model), the second is a refund (large model + approval path).
print(await handle_support_request("What is your return window?", "cust-99"))
print("---")
print(await handle_support_request("Where is my order A1002?", "cust-42"))
print("---")
# Repeat the first question -> served from cache.
print(await handle_support_request("What is your return window?", "cust-99"))
print("---")
print("Routing counters:", route_counters)

## 7. Cổng Đánh Giá

Đây là cổng phát hành từ bài học: một bộ dữ liệu kiểm tra ngoại tuyến đánh giá tác nhân, và việc triển khai chỉ tiến hành nếu tỷ lệ đạt vượt qua ngưỡng. Phương pháp điểm ở đây là một kiểm tra chồng chéo từ khóa đơn giản để giữ cho sổ ghi chép tự chứa; trong sản xuất bạn sẽ sử dụng một LLM làm giám khảo hoặc một bộ đánh giá khung (xem Bài học 10).


In [ ]:
TEST_CASES = [
    {"input": "How long do I have to return an item?", "expected": ["30 days", "refund"]},
    {"input": "How fast is standard shipping?", "expected": ["3-5", "business days"]},
    {"input": "What is the status of order A1001?", "expected": ["shipped", "A1001"]},
    {"input": "Do your electronics have a warranty?", "expected": ["12-month", "warranty"]},
]


def score_response(actual: str, expected_keywords: list[str]) -> float:
    actual_l = actual.lower()
    hits = sum(1 for kw in expected_keywords if kw.lower() in actual_l)
    return hits / len(expected_keywords)


async def evaluation_gate(test_cases: list[dict], threshold: float = 0.8) -> bool:
    passed = 0
    for case in test_cases:
        result = await support_agent.run(case["input"])
        s = score_response(result.text, case["expected"])
        status = "PASS" if s >= 0.5 else "FAIL"
        print(f"[{status}] {case['input']}  (score={s:.0%})")
        if s >= 0.5:
            passed += 1
    pass_rate = passed / len(test_cases)
    print(f"\nEvaluation pass rate: {pass_rate:.0%} (gate: {threshold:.0%})")
    return pass_rate >= threshold


gate_passed = await evaluation_gate(TEST_CASES, threshold=0.8)
print("\nDeploy allowed:" , gate_passed)

## Tổng Hợp: Một Phiên Bản Mô Phỏng

Ô bên dưới cho thấy toàn bộ vòng lặp mà bài học mô tả: chạy cổng đánh giá, và chỉ "triển khai" nếu nó vượt qua. Đây là mẫu bạn sẽ chạy trong CI trước khi đẩy một phiên bản agent lên Dịch vụ Agent Foundry.


In [ ]:
async def release(test_cases: list[dict]) -> None:
    print("Running pre-deployment evaluation gate...\n")
    if await evaluation_gate(test_cases, threshold=0.8):
        print("\n✅ Gate passed — promoting agent version to the Foundry Agent Service.")
    else:
        print("\n❌ Gate failed — release blocked. Fix the agent and re-run.")


await release(TEST_CASES)

## Tóm tắt

Bạn đã lắp ráp một đại lý hỗ trợ khách hàng sẵn sàng cho sản xuất với mọi mối quan tâm vận hành được kết nối:

- **Công cụ, RAG và bộ nhớ** cung cấp khả năng và bối cảnh cho đại lý.
- **Điều phối mô hình và bộ nhớ đệm** giữ cho độ trễ và chi phí được kiểm soát.
- **Phê duyệt của con người** bảo vệ các hành động rủi ro cao như hoàn tiền lớn.
- **Cửa chốt đánh giá** chặn các phiên bản xấu trước khi phát hành.
- **Theo dõi** làm cho mọi yêu cầu có thể quan sát được.

### Thách thức

Mở rộng đại lý này để:

1. **Hỗ trợ nhiều mô hình** — thêm một tầng "lý luận" thứ ba và điều phối các vấn đề/xử lý phàn nàn đến tầng này.
2. **Thêm các cửa chốt đánh giá** — mở rộng `TEST_CASES` để bao gồm các kịch bản phê duyệt hoàn tiền và xác nhận rằng cửa chốt bắt được các sự cố suy giảm.
3. **Thêm điều phối nhận biết chi phí** — theo dõi chi phí ước tính cho mỗi yêu cầu (nhỏ so với lớn so với bộ nhớ đệm) và in báo cáo chi phí sau một lô các câu truy vấn hỗn hợp.

Trong bài học tiếp theo, bạn sẽ đi theo hành trình ngược lại và chạy một đại lý hoàn toàn trên máy của bạn với Microsoft Foundry Local và Qwen.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Tuyên bố miễn trừ trách nhiệm**:
Tài liệu này đã được dịch bằng dịch vụ dịch thuật AI [Co-op Translator](https://github.com/Azure/co-op-translator). Mặc dù chúng tôi cố gắng đảm bảo độ chính xác, xin lưu ý rằng bản dịch tự động có thể chứa lỗi hoặc sai sót. Tài liệu gốc bằng ngôn ngữ gốc nên được coi là nguồn tin chính thức. Đối với thông tin quan trọng, nên sử dụng dịch vụ dịch thuật chuyên nghiệp bởi con người. Chúng tôi không chịu trách nhiệm về bất kỳ hiểu lầm hoặc giải thích sai nào phát sinh từ việc sử dụng bản dịch này.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
